# 01 – Exploration: Portuguese electricity load

Dataset: UCI ElectricityLoadDiagrams20112014 (CC BY 4.0), Trindade, A. (2015), https://doi.org/10.24432/C58C86

**Main findings** (written last, each backed by a chart below):

1. shape: (140256, 370) 
2. range: 2011-01-01 00:15:00 -> 2015-01-01 00:00:00
3. Duplicates exist and steps of 15min
4. Dtypes float32
5. 209MB

## 1. Load

In [4]:
from pathlib import Path

import pandas as pd

# VS Code runs notebooks from their own folder; fall back to the repo root otherwise.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "LD2011_2014.txt"
INTERIM = ROOT / "data" / "interim"

RAW.exists(), RAW

(True,
 WindowsPath('c:/Users/martim.oliveira/Desktop/OwnProjects/PraticalMLOps/MLOps/data/raw/LD2011_2014.txt'))

### Peek at the raw text before parsing

Rows have 371 columns, so only the start of each line is printed. Check the separator, the decimal mark, and whether the first timestamp is `00:00` or `00:15` (interval start vs end).

In [7]:
with RAW.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i in (0, 1, 90_000):
            print(line[:150])
        if i > 90_000:
            break


"";"MT_001";"MT_002";"MT_003";"MT_004";"MT_005";"MT_006";"MT_007";"MT_008";"MT_009";"MT_010";"MT_011";"MT_012";"MT_013";"MT_014";"MT_015";"MT_016";"MT
"2011-01-01 00:15:00";0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;
"2013-07-26 12:00:00";0;41,2517780938834;1,73761946133797;111,788617886179;69,5121951219512;238,095238095238;27,1339739966083;333,333333333333;57,6923


**Format notes:** _fill in from the output above: separator, decimal mark, first timestamp and what it means._

In [8]:
%%time
df = pd.read_csv(RAW, sep=";", decimal=",", index_col=0, parse_dates=True)
df.index.name = "timestamp"

CPU times: total: 3.83 s
Wall time: 3.98 s


### Sanity checks

Expect about 140,256 rows × 370 clients, a sorted index without duplicates, steps of exactly 15 minutes, and only `float64` columns. Any `object` column means the decimal setting is wrong.

In [9]:
print("shape:", df.shape)
print("range:", df.index.min(), "->", df.index.max())
print("sorted:", df.index.is_monotonic_increasing)
print("duplicate timestamps:", df.index.has_duplicates)
print("step sizes:", df.index.to_series().diff().value_counts().head(3).to_dict())
print("dtypes:", df.dtypes.value_counts().to_dict())
print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

shape: (140256, 370)
range: 2011-01-01 00:15:00 -> 2015-01-01 00:00:00
sorted: True
duplicate timestamps: False
step sizes: {Timedelta('0 days 00:15:00'): 140255}
dtypes: {dtype('float64'): 370}
memory: 416 MB


### Shrink and cache

`float32` halves the memory, and meter readings don't need more precision. The parquet copy loads in seconds, so later sessions and notebooks start from it instead of the 678 MB text file.

In [10]:
df = df.astype("float32")
print(f"memory: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB")

INTERIM.mkdir(parents=True, exist_ok=True)
df.to_parquet(INTERIM / "load_15min_raw.parquet")

memory: 209 MB


In [11]:
%%time
df = pd.read_parquet(INTERIM / "load_15min_raw.parquet")
df.shape

CPU times: total: 375 ms
Wall time: 140 ms


(140256, 370)